#### Extract youtube transcripts

Install library

In [1]:
# %pip install yt-dlp youtube-transcript-api

# https://github.com/jdepoix/youtube-transcript-api

In [4]:
# %pip install nbconvert

In [ ]:
import time
from datetime import date

import pandas as pd

from yt_dlp import YoutubeDL
from yt_dlp.utils import DownloadError

from youtube_transcript_api import (
    YouTubeTranscriptApi,
    TranscriptsDisabled,
    NoTranscriptFound,
)

print("imports OK")

Step 1: Paramters in one place

In [ ]:
channel_name = "https://www.youtube.com/@USATODAY"   # change this
start_date = date(2026, 11, 3)
end_date = date(2026, 11, 30)
lang = ["en"]
sleep = 1.0

print(channel_name, start_date, end_date)
print("Days in range:", (end_date - start_date).days)

Step 3: Date helper

In [ ]:
# Convert "20250314" -> date(2025, 3, 14).

def parse(raw):
    return date(int(raw[:4]), int(raw[4:6]), int(raw[6:]))

# test it
print(parse("20250314"))

Step 4: Get the list of videos IDs

In [ ]:
url = channel_name.rstrip("/") + "/videos"

with YoutubeDL({"extract_flat": "in_playlist", "quiet": True}) as ydl:
    playlist = ydl.extract_info(url, download=False)

ids = [e["id"] for e in playlist["entries"] if e.get("id")]

print("Total videos:", len(ids))
print("First 5 IDs:", ids[:5])

Step 5: Look at 1 video’s info before looping

In [ ]:
with YoutubeDL({"quiet": True, "no_warnings": True}) as ydl:
    info = ydl.extract_info(
        f"https://www.youtube.com/watch?v={ids[0]}",
        download=False,
        process=False,
    )

print(info["title"])
print(info["upload_date"], "parsed to", parse(info["upload_date"]))

In [ ]:
fields = ["view_count", "like_count", "comment_count", "duration",
          "channel_follower_count", "categories", "tags"]

for f in fields:
    print(f, ":", info.get(f))

Step 6: Filter by date

In [ ]:
found = []

with YoutubeDL({"quiet": True, "no_warnings": True}) as ydl:
    for vid in ids:
        try:
            info = ydl.extract_info(
                f"https://www.youtube.com/watch?v={vid}",
                download=False,
                process=False,
            )
        except DownloadError as err:
            print("skip", vid, err)
            continue

        if not info.get("upload_date"):
            continue
        d = parse(info["upload_date"])

        if d > end_date:
            print("too new: ", d, info["title"])
            continue
        if d < start_date:
            print("too old: ", d, "-> stopping")
            break

        found.append({
            "video_id": vid,
            "date": d,
            "title": info["title"],
            "views": info.get("view_count"),
            "likes": info.get("like_count"),
            "comments": info.get("comment_count"),
            "duration_sec": info.get("duration"),
            "subscribers": info.get("channel_follower_count")            })
        print("iin range:", d, info["title"])
        time.sleep(sleep)

print("Videos in range:", len(found))

In [ ]:
ytt = YouTubeTranscriptApi()
test = ytt.fetch(found[0]["video_id"], languages=lang)
print("Segments:", len(test))
print("Auto-generated?", test.is_generated)
print(test[0])
print(test[1])

In [ ]:
segments = []
videos = []

for v in found:
    try:
        t = ytt.fetch(v["video_id"], languages=lang)
    except (TranscriptsDisabled, NoTranscriptFound):
        print("no transcript:", v["title"])
        continue
    except Exception as err:
        print("error:", v["video_id"], err)
        continue

    for s in t:
        segments.append({
            **v,
            "start": s.start,
            "duration": s.duration,
            "text": s.text,
            "url": f"https://youtu.be/{v['video_id']}?t={int(s.start)}",
        })

    videos.append({
        **v,
        "is_generated": t.is_generated,
        "text": " ".join(s.text for s in t),
    })
    print("got:", v["date"], v["title"])
    time.sleep(sleep)

print("\nSegments:", len(segments), "| Videos:", len(videos))

In [ ]:
seg_df = pd.DataFrame(segments)
vid_df = pd.DataFrame(videos)

vid_df.head()

In [ ]:
word = "election"   # change this
hits = seg_df[seg_df["text"].str.contains(word, case=False)]

print(len(hits), "mentions")
hits.head(10)

How to do this using YouTube API:

1. Go to the Google Cloud Console (console.cloud.google.com) and create a project.
2. Under APIs & Services → Library, search for “YouTube Data API v3” and click Enable.
3. Under APIs & Services → Credentials, click Create credentials → API key.